https://openai.github.io/openai-agents-python/

In [36]:
!pip install openai-agents -qq

In [77]:
import nest_asyncio
nest_asyncio.apply()
from rich import  print

In [38]:
from openai import AsyncAzureOpenAI, AzureOpenAI
from google.colab import userdata
# client = AzureOpenAI(api_key=userdata.get('AZURE_API_KEY'), azure_endpoint=userdata.get('AZURE_BASE_URL'), api_version=userdata.get('AZURE_API_VERSION'))
async_client = AsyncAzureOpenAI(api_key=userdata.get('AZURE_API_KEY'), azure_endpoint=userdata.get('AZURE_BASE_URL'), api_version=userdata.get('AZURE_API_VERSION'))

In [39]:
# https://openai.github.io/openai-agents-python/config/
from agents import set_default_openai_client, set_default_openai_api, set_tracing_disabled
# set_default_openai_client(client)
# set_default_openai_client(async_client)
set_default_openai_client(async_client, use_for_tracing=False)
set_default_openai_api("chat_completions")
set_tracing_disabled(True)

In [40]:
from agents import Agent, Runner

agent = Agent(name="Assistant", instructions="You are a helpful assistant",
            #   model='gpt-4o' # default
              )

result = Runner.run_sync(agent, "Write a haiku about recursion in programming.")
print(result.final_output)

Code echoes within,
Looping through its mirrored depths—
Infinite returns.

In [41]:
from agents import Agent, ModelSettings, function_tool

@function_tool
def get_weather(city: str) -> str:
    return f"The weather in {city} is sunny"

haiku_agent = Agent(
    name="Haiku agent",
    instructions="Always respond in haiku form",
    # model="gpt-4o",
    tools=[get_weather],
)

In [42]:
from pydantic import BaseModel
from agents import Agent


class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

calendar_agent = Agent(
    name="Calendar extractor",
    instructions="Extract calendar events from text",
    output_type=CalendarEvent,
    model='gpt-4o-2'
)

In [43]:
# Handoffs are sub-agents
triage_agent = Agent(
    name="Triage agent",
    instructions=(
        "Help the user with their questions."
        "If they ask about booking, handoff to the booking agent."
        "If they ask about refunds, handoff to the refund agent."
    ),
    handoffs=[haiku_agent, calendar_agent],
)

In [102]:
from agents import Agent, Runner, AsyncOpenAI, OpenAIChatCompletionsModel
import asyncio

spanish_agent = Agent(
    name="Spanish agent",
    instructions="You only speak Spanish.",
    model="gpt-4o-2",
)

english_agent = Agent(
    name="English agent",
    instructions="You only speak English",
    model="gpt-4o-2",
    # model=OpenAIChatCompletionsModel(
    #     model="gpt-4o",
    #     openai_client=AsyncOpenAI()
    # ),
)

triage_agent = Agent(
    name="Triage agent",
    instructions="Handoff to the appropriate agent based on the language of the request.",
    handoffs=[spanish_agent, english_agent],
    model="gpt-4o-2",
)

result = await Runner.run(triage_agent, input="Hola, ¿cómo estás? ")
print(result.final_output)

¡Hola! Estoy bien, gracias. ¿Y tú cómo estás?

In [45]:
from dataclasses import dataclass

@dataclass
class UserContext:
  uid: str
  is_pro_user: bool

In [46]:
#  Dynamic instructions
from agents import RunContextWrapper
def dynamic_instructions(
    context: RunContextWrapper[UserContext], agent: Agent[UserContext]
) -> str:
    return f"The user's name is {context.context.name}. Help them with their questions."


agent = Agent[UserContext](
    name="Triage agent",
    instructions=dynamic_instructions,
)

In [47]:
pirate_agent = Agent(
    name="Pirate",
    instructions="Write like a pirate",
    model="o3-mini",
)

robot_agent = pirate_agent.clone(
    name="Robot",
    instructions="Write like a robot",
)

In [48]:
from agents import Agent, Runner, AsyncOpenAI, OpenAIChatCompletionsModel
import asyncio

spanish_agent = Agent(
    name="Spanish agent",
    instructions="You only speak Spanish.",
    # model="o3-mini",
)

english_agent = Agent(
    name="English agent",
    instructions="You only speak English",
    # model=OpenAIChatCompletionsModel(
    #     model="gpt-4o",
    #     openai_client=AsyncOpenAI()
    # ),
)

triage_agent = Agent(
    name="Triage agent",
    instructions="Handoff to the appropriate agent based on the language of the request.",
    handoffs=[spanish_agent, english_agent],
    # model="gpt-3.5-turbo",
)

result = await Runner.run(triage_agent, input="Hola, ¿cómo estás?")
print(result.final_output)

Hola, estoy bien, gracias. ¿Y tú, cómo estás?

In [112]:
# from agents import Agent, Runner, trace
# agent = Agent(name="Assistant", instructions="Reply very concisely.")
# with trace(workflow_name="Conversation", group_id="..."):
#     # First turn
#     result = await Runner.run(agent, "What city is the Golden Gate Bridge in?")
#     print(result.final_output)
#     # San Francisco

#     # Second turn
#     new_input = result.to_input_list() + [{"role": "user", "content": "What state is it in?"}]
#     result = await Runner.run(agent, new_input)
#     print(result.final_output)
#     # California

In [81]:
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner

agent = Agent(
    name="Joker",
    instructions="You are a helpful assistant.",
)

result = Runner.run_streamed(agent, input="Please tell me 5 jokes.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, sep=" ",end="", flush=True)


Of

course

!

Here

are

...

smile

to

your

face

!

In [51]:
joker_agent = Agent(
    name="Joker",
    instructions="You are a helpful assistant.",
)
Runner.run_sync(joker_agent, "Please tell me a joke.").final_output

"Sure, here's a light-hearted joke for you:\n\nWhy don't scientists trust atoms?\n\nBecause they make up everything!"

In [52]:
import asyncio
import random
from agents import Agent, ItemHelpers, Runner, function_tool

joker_agent = Agent(
    name="Joker",
    instructions="You are a helpful assistant.",
)
@function_tool
def generate_joke() -> str:
    return Runner.run_sync(joker_agent, "Please tell me a joke.").final_output

@function_tool
def how_many_jokes() -> int:
    return random.randint(1, 10)

agent = Agent(
    name="Joker",
    instructions="First call the `how_many_jokes` tool, then tell that many jokes.",
    tools=[how_many_jokes, generate_joke],
)

result = Runner.run_streamed(
    agent,
    input="Hello, generate jokes",
)
print("=== Run starting ===")

async for event in result.stream_events():
    # We'll ignore the raw responses event deltas
    if event.type == "raw_response_event":
        continue
    # When the agent updates, print that
    elif event.type == "agent_updated_stream_event":
        print(f"Agent updated: {event.new_agent.name}")
        continue
    # When items are generated, print them
    elif event.type == "run_item_stream_event":
        if event.item.type == "tool_call_item":
            print("-- Tool was called")
        elif event.item.type == "tool_call_output_item":
            print(f"-- Tool output: {event.item.output}")
        elif event.item.type == "message_output_item":
            print(f"-- Message output:\n {ItemHelpers.text_message_output(event.item)}")
        else:
            pass  # Ignore other event types

print("=== Run complete ===")

=== Run starting ===

Agent updated: Joker

-- Tool was called

-- Tool output: 7

-- Tool was called

...

-- Tool output: Sure, here's a light-hearted joke for you:

Why don't scientists trust atoms?

Because they make up everything!

-- Tool output: Sure, here's one for you:

Why don't scientists trust atoms?

Because they make up everything!

-- Tool output: Sure, here's a joke for you:

Why did the scarecrow win an award?

Because he was outstanding in his field!

-- Message output:
 Here are 7 jokes for you:

1. Why don't skeletons fight each other?  
   They don't have the guts!

2. Why don't skeletons fight each other?  
   They don't have the guts!

3. Why don't scientists trust atoms?  
   Because they make up everything!

4. Why did the scarecrow win an award?  
   Because he was outstanding in his field!

5. Why don't scientists trust atoms?  
   Because they make up everything!

6. Why don't scientists trust atoms?  
   Because they make up everything!

7. Why did the scarecrow win an award?  
   Because he was outstanding in his field!

=== Run complete ===

In [63]:
# from agents import Agent, FileSearchTool, Runner, WebSearchTool

# agent = Agent(
#     name="Assistant",
#     tools=[
#         WebSearchTool(),
#         FileSearchTool(
#             max_num_results=3,
#             vector_store_ids=["VECTOR_STORE_ID"],
#         ),
#     ],
# )

# result = Runner.run_sync(agent, "Which coffee shop should I go to, taking into account my preferences and the weather today in SF?")
# print(result.final_output)

In [64]:
import json

from typing_extensions import TypedDict, Any

from agents import Agent, FunctionTool, RunContextWrapper, function_tool


class Location(TypedDict):
    lat: float
    long: float

@function_tool
async def fetch_weather(location: Location) -> str:

    """Fetch the weather for a given location.

    Args:
        location: The location to fetch the weather for.
    """
    # In real life, we'd fetch the weather from a weather API
    return "sunny"


@function_tool(name_override="fetch_data")
def read_file(ctx: RunContextWrapper[Any], path: str, directory: str | None = None) -> str:
    """Read the contents of a file.

    Args:
        path: The path to the file to read.
        directory: The directory to read the file from.
    """
    # In real life, we'd read the file from the file system
    return "<file contents>"


agent = Agent(
    name="Assistant",
    tools=[fetch_weather, read_file],
)

for tool in agent.tools:
    if isinstance(tool, FunctionTool):
        print(tool.name)
        print(tool.description)
        print(json.dumps(tool.params_json_schema, indent=2))
        print()

fetch_weather

Fetch the weather for a given location.

{
  "$defs": {
    "Location": {
      "properties": {
        "lat": {
          "title": "Lat",
          "type": "number"
        },
        "long": {
          "title": "Long",
          "type": "number"
        }
      },
      "required": [
        "lat",
        "long"
      ],
      "title": "Location",
      "type": "object",
      "additionalProperties": false
    }
  },
  "properties": {
    "location": {
      "description": "The location to fetch the weather for.",
      "properties": {
        "lat": {
          "title": "Lat",
          "type": "number"
        },
        "long": {
          "title": "Long",
          "type": "number"
        }
      },
      "required": [
        "lat",
        "long"
      ],
      "title": "Location",
      "type": "object",
      "additionalProperties": false
    }
  },
  "required": [
    "location"
  ],
  "title": "fetch_weather_args",
  "type": "object",
  "additionalProperties": false
}

fetch_data

Read the contents of a file.

{
  "properties": {
    "path": {
      "description": "The path to the file to read.",
      "title": "Path",
      "type": "string"
    },
    "directory": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "description": "The directory to read the file from.",
      "title": "Directory"
    }
  },
  "required": [
    "path",
    "directory"
  ],
  "title": "fetch_data_args",
  "type": "object",
  "additionalProperties": false
}

In [65]:
from typing import Any

from pydantic import BaseModel

from agents import RunContextWrapper, FunctionTool



def do_some_work(data: str) -> str:
    return "done"


class FunctionArgs(BaseModel):
    username: str
    age: int


async def run_function(ctx: RunContextWrapper[Any], args: str) -> str:
    parsed = FunctionArgs.model_validate_json(args)
    return do_some_work(data=f"{parsed.username} is {parsed.age} years old")


tool = FunctionTool(
    name="process_user",
    description="Processes extracted user data",
    params_json_schema=FunctionArgs.model_json_schema(),
    on_invoke_tool=run_function,
)

In [66]:
from agents import Agent, Runner
import asyncio

spanish_agent = Agent(
    name="Spanish agent",
    instructions="You translate the user's message to Spanish",
)

french_agent = Agent(
    name="French agent",
    instructions="You translate the user's message to French",
)

orchestrator_agent = Agent(
    name="orchestrator_agent",
    instructions=(
        "You are a translation agent. You use the tools given to you to translate."
        "If asked for multiple translations, you call the relevant tools."
    ),
    tools=[
        spanish_agent.as_tool(
            tool_name="translate_to_spanish",
            tool_description="Translate the user's message to Spanish",
        ),
        french_agent.as_tool(
            tool_name="translate_to_french",
            tool_description="Translate the user's message to French",
        ),
    ],
)

result = Runner.run_sync(orchestrator_agent, input="Say 'Hello, how are you?' in Spanish.")
print(result.final_output)

"Hello, how are you?" in Spanish is: "Hola, ¿cómo estás?"

# Handoffs

In [82]:
from agents import Agent, handoff

billing_agent = Agent(name="Billing agent")
refund_agent = Agent(name="Refund agent")


triage_agent = Agent(name="Triage agent", handoffs=[billing_agent, handoff(refund_agent)])

In [83]:
from agents import Agent, handoff, RunContextWrapper

def on_handoff(ctx: RunContextWrapper[None]):
    print("Handoff called")

agent = Agent(name="My agent")

handoff_obj = handoff(
    agent=agent,
    on_handoff=on_handoff,
    tool_name_override="custom_handoff_tool",
    tool_description_override="Custom description",
)

In [69]:
from pydantic import BaseModel

from agents import Agent, handoff, RunContextWrapper

class EscalationData(BaseModel):
    reason: str

async def on_handoff(ctx: RunContextWrapper[None], input_data: EscalationData):
    print(f"Escalation agent called with reason: {input_data.reason}")

agent = Agent(name="Escalation agent")

handoff_obj = handoff(
    agent=agent,
    on_handoff=on_handoff,
    input_type=EscalationData,
)

In [70]:
from agents import Agent, handoff
from agents.extensions import handoff_filters

agent = Agent(name="FAQ agent")

handoff_obj = handoff(
    agent=agent,
    input_filter=handoff_filters.remove_all_tools,
)

In [71]:
from agents import Agent
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX

billing_agent = Agent(
    name="Billing agent",
    instructions=f"""{RECOMMENDED_PROMPT_PREFIX}
    <Fill in the rest of your prompt here>.""",
)

In [72]:
print(RECOMMENDED_PROMPT_PREFIX)

# System context
You are part of a multi-agent system called the Agents SDK, designed to make agent coordination and execution easy.
Agents uses two primary abstraction: **Agents** and **Handoffs**. An agent encompasses instructions and tools and 
can hand off a conversation to another agent when appropriate. Handoffs are achieved by calling a handoff function,
generally named `transfer_to_<agent_name>`. Transfers between agents are handled seamlessly in the background; do 
not mention or draw attention to these transfers in your conversation with the user.

In [73]:
from agents import Agent, Runner, trace

agent = Agent(name="Joke generator", instructions="Tell funny jokes.")

with trace("Joke workflow"):
    first_result = await Runner.run(agent, "Tell me a joke")
    second_result = await Runner.run(agent, f"Rate this joke: {first_result.final_output}")
    print(f"Joke: {first_result.final_output}")
    print(f"Rating: {second_result.final_output}")

Joke: Sure, here's one for you:

Why don't scientists trust atoms?

Because they make up everything!

Rating: I'd give that joke a solid 8 out of 10! It's a classic science joke with a clever play on words. Nicely 
done!

In [86]:
import asyncio
from dataclasses import dataclass

from agents import Agent, RunContextWrapper, Runner, function_tool

@dataclass
class UserInfo:
    name: str
    uid: int

@function_tool
async def fetch_user_age(wrapper: RunContextWrapper[UserInfo]) -> str:
    if wrapper.context.uid == 123:
        return f"User {wrapper.context.name} is 47 years old"
    else:
        return f"User has 21 years old"

user_info = UserInfo(name="John", uid=123)

agent = Agent[UserInfo](
    name="Assistant",
    tools=[fetch_user_age],
)

# result = await Runner.run(
result = Runner.run_sync(
    starting_agent=agent,
    input="What is the age of the user?",
    context=user_info,
)

print(result.final_output)

The user, John, is 47 years old.

# Guardrails

In [89]:
from pydantic import BaseModel
from agents import (
    Agent,
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    RunContextWrapper,
    Runner,
    TResponseInputItem,
    input_guardrail,
)

class MathHomeworkOutput(BaseModel):
    is_math_homework: bool
    reasoning: str

guardrail_agent = Agent(
    name="Guardrail check",
    model="gpt-4o-2",
    instructions="Check if the user is asking you to do their math homework.",
    output_type=MathHomeworkOutput,
)


@input_guardrail
async def math_guardrail(
    ctx: RunContextWrapper[None], agent: Agent, input: str | list[TResponseInputItem]
) -> GuardrailFunctionOutput:
    result = await Runner.run(guardrail_agent, input, context=ctx.context)

    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=result.final_output.is_math_homework,
    )


agent = Agent(
    name="Customer support agent",
    instructions="You are a customer support agent. You help customers with their questions.",
    input_guardrails=[math_guardrail],
)

try:
    await Runner.run(agent, "Hello, can you help me solve for x: 2x + 3 = 11?")
    print("Guardrail didn't trip - this is unexpected")

except InputGuardrailTripwireTriggered:
    print("Math homework guardrail tripped")

Math homework guardrail tripped

In [92]:
from pydantic import BaseModel
from agents import (
    Agent,
    GuardrailFunctionOutput,
    OutputGuardrailTripwireTriggered,
    RunContextWrapper,
    Runner,
    output_guardrail,
)
class MessageOutput(BaseModel):
    response: str

class MathOutput(BaseModel):
    is_math: bool
    reasoning: str

guardrail_agent = Agent(
    model="gpt-4o-2",
    name="Guardrail check",
    instructions="Check if the output includes any math.",
    output_type=MathOutput,
)

@output_guardrail
async def math_guardrail(
    ctx: RunContextWrapper, agent: Agent, output: MessageOutput
) -> GuardrailFunctionOutput:
    result = await Runner.run(guardrail_agent, output.response, context=ctx.context)

    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=result.final_output.is_math,
    )

agent = Agent(
    model="gpt-4o-2",
    name="Customer support agent",
    instructions="You are a customer support agent. You help customers with their questions.",
    output_guardrails=[math_guardrail],
    output_type=MessageOutput,
)

try:
    await Runner.run(agent, "Hello, can you help me solve for x: 2x + 3 = 11?")
    print("Guardrail didn't trip - this is unexpected")

except OutputGuardrailTripwireTriggered:
    print("Math output guardrail tripped")

Math output guardrail tripped

In [111]:
from agents import ModelProvider, Model, OpenAIChatCompletionsModel, RunConfig

class CustomModelProvider(ModelProvider):
    def get_model(self, model_name: str | None) -> Model:
        return OpenAIChatCompletionsModel(model='gpt-4o-2', openai_client=async_client)


CUSTOM_MODEL_PROVIDER = CustomModelProvider()
agent = Agent(name="Assistant", instructions="You only respond in haikus.", tools=[get_weather])

result = await Runner.run(
    agent,
    "What's the weather in Tokyo?",
    run_config=RunConfig(model_provider=CUSTOM_MODEL_PROVIDER),
)
print(result.final_output)

Tokyo's sky is clear,  
Sunshine warms the bustling streets,  
A bright day awaits.